
# TEL351 - Tarea 1: Agentes periodistas y generación automatizada de noticias

Nombre: Pedro Arce

## Objetivo
En esta tarea desarrollarás un sistema que scrapea noticias de un sitio web público, crea agentes automatizados que generan nuevas versiones de estas noticias, y luego presenta el resultado en un sitio web HTML.

## Detalle

- Recolectar al menos 100 noticias de un sitio web de libre acceso utilizando web scraping.

- Procesar los datos obtenidos y almacenarlos de manera estructurada.

- Diseñar 5 agentes, cada uno con una personalidad distinta, que generen noticias basadas en las originales o en su interpretación.

- Construir un sitio web HTML estático que simule ser un portal de noticias con las publicaciones generadas por los agentes.

## Parte 1: Web Scraping de noticias

In [1]:

# Importar librerías necesarias
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
import json
import unicodedata
from urllib.parse import urlparse
from datetime import datetime
import random
import os
from collections import defaultdict, Counter
import spacy
from textblob import TextBlob
import dateutil.parser as parser
import torch
# pip install transformers sentence-transformers torch
# pip install jinja2
###Descomentar en caso de no tener instalada alguna de las librerias...
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer, util
from jinja2 import Template



### Instrucciones:
- Elige un sitio de noticias público (ej. [BioBioChile](https://www.biobiochile.cl/), [Emol](https://www.emol.com/), [CNN Chile](https://www.cnnchile.com/), [BBC Mundo](https://www.bbc.com/mundo)) que:
    - No requiera login.
    - Tenga estructura consistente (mismo layout para artículos).
    - Permita scraping ético (consulta `robots.txt` si es necesario).
- Usa Python con `requests` + `BeautifulSoup`, o bien con `selenium` si es un sitio dinámico.
- Extrae y almacena para **al menos 100 artículos**:
    - Título.
    - Fecha.
    - Cuerpo completo de la noticia.
    - URL original.
    - Categoría o sección (si es posible).
- Guarda los datos en un archivo `.json`, `.csv` o `.pkl` estructurado.
- Asegúrate de limpiar caracteres extraños, eliminar etiquetas HTML y normalizar el texto (acentos, espacios, etc).

In [2]:
# Código de scraping aquí

#Sitio objetivo (CNN Chile)
BASE = "https://www.cnnchile.com"
ROBOTS = f"{BASE}/robots.txt"
SITEMAP_INDEX = f"{BASE}/_files/sitemaps/sitemap_index.xml"
SITEMAP_NEWS  = f"{BASE}/_files/sitemaps/sitemap_news.xml"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; ProyScraper/1.0; +https://tu-dominio-ejemplo.cl/)",
    "Accept-Language": "es,es-CL;q=0.9,en;q=0.8"
}

CATEGORY_WHITELIST = {
    "pais","mundo","negocios","cultura","deportes","servicios","bits","miradas","opinion","cnn-data","futuro-360"
}

def fetch(url, retries=3, backoff=2.0, timeout=15):
    """DESCARGA ROBUSTA → devuelve BYTES (no texto)."""
    for i in range(retries):
        try:
            r = requests.get(url, headers=HEADERS, timeout=timeout)
            if 200 <= r.status_code < 300:
                return r.content  #clave para evitar mojibake
            time.sleep(backoff * (i + 1))
        except requests.RequestException:
            time.sleep(backoff * (i + 1))
    return None

def normalize_text(s: str) -> str:
    if not s:
        return ""
    s = unicodedata.normalize("NFKC", s)
    s = s.replace("\xa0", " ").replace("\u200b", "")
    s = re.sub(r"\s+", " ", s, flags=re.M)
    return s.strip()

def is_article_url(url: str) -> bool:
    try:
        path = urlparse(url).path.strip("/")
        first = path.split("/", 1)[0].lower()
        if first in CATEGORY_WHITELIST:
            return True
        if first in {"tag","category","programas","en-vivo","search"}:
            return False
        return False
    except Exception:
        return False

In [3]:
###2. Sitemaps => recolectar URLs (>= 100)

def parse_xml_locs(xml_bytes) -> list:
    """Extrae <loc> desde XML (recibe BYTES)."""
    if not xml_bytes:
        return []
    soup = BeautifulSoup(xml_bytes, "xml")
    return [loc.get_text(strip=True) for loc in soup.find_all("loc")]

def get_sitemap_months():
    xml = fetch(SITEMAP_INDEX)
    months = parse_xml_locs(xml)
    months = [m for m in months if "/_files/sitemaps/" in m]
    months.sort(reverse=True)  #mas recientes primero
    return months

def get_news_urls(limit=200):
    urls = []

    news_xml = fetch(SITEMAP_NEWS)
    urls.extend(parse_xml_locs(news_xml))

    if len(urls) < limit:
        for sm in get_sitemap_months():
            xml = fetch(sm)
            urls.extend(parse_xml_locs(xml))
            if len(urls) >= limit * 2:
                break

    seen = set()
    filtered = []
    for u in urls:
        if not u or not u.startswith("http"):
            continue
        if is_article_url(u) and u not in seen:
            seen.add(u)
            filtered.append(u)

    filtered = [u for u in filtered if not re.search(r"\.(jpg|jpeg|png|webp)(\?|$)", u, re.I)]
    return filtered[:max(limit, 120)]

article_urls = get_news_urls(limit=200)
len(article_urls), article_urls[:5]

(185,
 ['https://www.cnnchile.com/cultura/como-pedro-pascal-se-convirtio-en-el-actor-mas-cotizado-de-hollywood-en-2025_20250831/',
  'https://www.cnnchile.com/pais/delegado-duran-no-suspendio-partido-la-u-colo-colo-muerte-hincha_20250831/',
  'https://www.cnnchile.com/pais/alcalde-independencia-mala-decision-jugar-supercopa-santa-laura-muerte-hincha-monumental_20250831/',
  'https://www.cnnchile.com/deportes/vicente-pizarro-tras-victoria-de-colo-colo-en-el-superclasico-con-nuestra-gente-nos-tenemos-que-hacer-fuertes_20250831/',
  'https://www.cnnchile.com/pais/se-registra-incendio-estructural-en-una-vivienda-en-cerro-yungay-en-valparaiso_20250831/'])

In [4]:
###3. Parser de artículos (título, fecha, cuerpo, url, categoría)

DATE_PATTERNS = [
    ("%Y-%m-%dT%H:%M:%S%z", r"\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}(?:Z|[+-]\d{2}:\d{2})"),
    ("%d.%m.%Y / %H:%M", r"\b\d{2}\.\d{2}\.\d{4}\s*/\s*\d{2}:\d{2}\b"),
]

BAD_PREFIXES = (
    "Lee también", "LEA TAMBIÉN", "Relacionado", "RELACIONADO",
    "Video:", "VIDEO:", "Vea también", "VEA TAMBIÉN",
    "Compartir", "DESTACAMOS", "LO ÚLTIMO", "PROGRAMAS DE TV"
)

def parse_date_from_meta(soup: BeautifulSoup) -> str | None:
    metas = [
        ("meta", {"property": "article:published_time"}),
        ("meta", {"name": "article:published_time"}),
        ("meta", {"property": "og:updated_time"}),
        ("meta", {"name": "date"}),
        ("meta", {"itemprop": "datePublished"}),
        ("time",  {"itemprop": "datePublished"}),
    ]
    for tag_name, attrs in metas:
        tag = soup.find(tag_name, attrs=attrs)
        if tag:
            content = tag.get("content") or tag.get_text(strip=True)
            if content:
                iso = content.replace("Z", "+00:00")
                try:
                    return datetime.fromisoformat(iso).isoformat()
                except Exception:
                    pass
    return None

def parse_date_fallback(page_text: str) -> str | None:
    for fmt, regex in DATE_PATTERNS:
        m = re.search(regex, page_text)
        if m:
            date_str = m.group(0)
            try:
                dt = datetime.strptime(date_str, fmt)
                return dt.isoformat()
            except Exception:
                continue
    return None

def get_category_from_url(url: str) -> str:
    path = urlparse(url).path.strip("/")
    return (path.split("/", 1)[0] if path else "").lower()

def extract_main_paragraphs(soup: BeautifulSoup) -> list[str]:
    candidates = []
    for tag in soup.find_all(["article", "section", "div"]):
        ps = tag.find_all("p")
        if len(ps) >= 3:
            candidates.append((len(ps), tag))
    candidates.sort(key=lambda x: x[0], reverse=True)

    paragraphs = []
    if candidates:
        _, best = candidates[0]
        for p in best.find_all("p"):
            txt = normalize_text(p.get_text(" ", strip=True))
            if not txt:
                continue
            if any(txt.startswith(bad) for bad in BAD_PREFIXES):
                continue
            if len(txt) < 25 and re.search(r"^([A-ZÁÉÍÓÚÑ# ]{3,}|[A-ZÁÉÍÓÚÑ0-9 ]{3,})$", txt):
                continue
            paragraphs.append(txt)
    return paragraphs

def parse_article(url: str) -> dict | None:
    html_bytes = fetch(url)
    if not html_bytes:
        return None

    soup = BeautifulSoup(html_bytes, "html.parser")

    h1 = soup.find("h1")
    title = normalize_text(h1.get_text(strip=True)) if h1 else ""

    date_iso = parse_date_from_meta(soup)
    if not date_iso:
        page_text = soup.get_text("\n", strip=True)
        date_iso = parse_date_fallback(page_text)

    body_paragraphs = extract_main_paragraphs(soup)
    body = normalize_text(" ".join(body_paragraphs))

    categoria = get_category_from_url(url)

    #Fallback JSON-LD para completar título/cuerpo/fecha
    if not title or not body or len(body) < 200:
        for script in soup.find_all("script", type="application/ld+json"):
            try:
                data = json.loads(script.string or "")
                items = [data] if isinstance(data, dict) else (data if isinstance(data, list) else [])
                for item in items:
                    if isinstance(item, dict) and item.get("@type") in {"NewsArticle","Article"}:
                        title = title or normalize_text(item.get("headline",""))
                        body = body or normalize_text(item.get("articleBody",""))
                        if not date_iso and item.get("datePublished"):
                            try:
                                iso = item["datePublished"].replace("Z","+00:00")
                                date_iso = datetime.fromisoformat(iso).isoformat()
                            except Exception:
                                pass
            except Exception:
                continue

    if not title or not body or len(body) < 200:
        return None

    return {
        "titulo": title,
        "fecha": date_iso or "",
        "cuerpo": body,
        "url": url,
        "categoria": categoria,
    }

In [5]:
###4. Ejecutar scraping (>=100), validar y guardar

def polite_sleep():
    time.sleep(0.7 + random.random() * 0.5)

records = []
for i, url in enumerate(article_urls):
    art = parse_article(url)
    if art:
        records.append(art)
    if (i+1) % 20 == 0:
        print(f"Procesadas {i+1} URLs, válidas: {len(records)}")
    polite_sleep()
    if len(records) >= 130:  #sobrecolectar
        break

df = pd.DataFrame.from_records(records)
df = df.drop_duplicates(subset=["url"]).dropna(subset=["titulo","cuerpo"])
print(df.shape)
df.head(3)

Procesadas 20 URLs, válidas: 20
Procesadas 40 URLs, válidas: 40
Procesadas 60 URLs, válidas: 60
Procesadas 80 URLs, válidas: 80
Procesadas 100 URLs, válidas: 100
Procesadas 120 URLs, válidas: 120
(130, 5)


,titulo,fecha,cuerpo,url,categoria
0,Cómo Pedro Pascal se convirtió en el actor más...,2025-08-31T18:38:00,Con una carrera que comenzó hace más de 25 año...,https://www.cnnchile.com/cultura/como-pedro-pa...,cultura
1,Durán explica por qué no se suspendió el parti...,2025-08-31T18:36:00,"Según explicó la autoridad, un análisis de Car...",https://www.cnnchile.com/pais/delegado-duran-n...,pais
2,Tras muerte de hincha en Monumental: Alcalde d...,2025-08-31T18:08:00,El jefe comunal advirtió que actualmente no ex...,https://www.cnnchile.com/pais/alcalde-independ...,pais


In [6]:
###5. Limpieza final y exportación

df["titulo"]   = df["titulo"].apply(normalize_text)
df["cuerpo"]   = df["cuerpo"].apply(normalize_text)
df["categoria"] = df["categoria"].str.lower().str.strip()

#Filtro de seguridad
df = df[df["cuerpo"].str.len() >= 400].copy()
df.reset_index(drop=True, inplace=True)

print("Total artículos listos:", len(df))
assert len(df) >= 100, "No se alcanzaron 100 artículos. Amplía el rango o relaja filtros."

CSV_OUT = "datos/cnnchile_noticias.csv"

#Crear carpeta 'datos' si no existe
if not os.path.exists('datos'):
    os.makedirs('datos')
    print("Carpeta 'datos' creada exitosamente")

#BOM para compatibilidad Excel/Windows
df.to_csv(CSV_OUT, index=False, encoding="utf-8-sig")

print(f"Archivos guardados en: {CSV_OUT}")

Total artículos listos: 130
Carpeta 'datos' creada exitosamente
Archivos guardados en: datos/cnnchile_noticias.csv


## Parte 2: Diseño de Agentes Generadores de Contenido


### Instrucciones:
- Diseña **5 agentes con personalidades distintas**.
- Cada agente debe generar al menos 10 noticias basadas en el corpus extraído.
- Cada agente debe tener un comportamiento **automatizado y único** al generar contenido. Por ejemplo:

| Agente                 | Personalidad           | Comportamiento esperado                                                                                                                           |
| ---------------------- | ---------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------- |
| 1. Aleatorio           | "Publicador impulsivo" | Elige noticias al azar, publica un extracto sin cambios.                                                                                          |
| 2. Imitador            | "Papagayo"             | Copia noticias, pero cambia algunas palabras por sinónimos usando NLP básico.                                                                     |
| 3. Seleccionador       | "Curador temático"     | Filtra noticias de una temática y publica resúmenes cortos.                                                                                       |
| 4. Reescritor creativo | "Periodista junior"    | Reescribe noticias con un estilo personal (con reglas simples de cambio de redacción).                                                            |
| 5. Generador semántico | "Periodista LLM"       | Usa un modelo de lenguaje (`transformers`, `GPT`, `LLaMA`, etc.) para redactar nuevas versiones de las noticias originales, interpretándolas. |


In [7]:
# Define cada agente

agentes_df = pd.DataFrame({
    'Agente': [
        'Titulares Impactantes Contextuales',
        'Analista de Datos Estadísticos',
        'Hashtagero Estratégico',
        'Preguntador Curioso',
        'Analogador Creativo'
    ],
    'Personalidad': [
        'Cazador de clics con criterio',
        'Científico de datos preciso',
        'Community manager viral',
        'Interviewer inquisitivo',
        'Traductor de realidades'
    ],
    'Comportamiento_esperado': [
        'Genera titulares clickbait ajustados a la categoría usando bancos de frases por temática. Nunca sensacionaliza tragedias; en esos casos usa un titular neutro. Mantiene cifras/fechas intactas y añade un breve gancho de intriga. Produce ≥10 piezas por categoría.',
        'Extrae cifras, porcentajes y datos numéricos de las noticias. Crea resúmenes estadísticos.',
        'Identifica palabras clave y términos trending en cada noticia. Genera combinaciones de hashtags por categoría temática. Añade hashtags de ubicación cuando son relevantes. Incluye hashtags de emociones según el tono de la noticia. Crea mixes de hashtags serios y creativos.',
        'Extrae 3-5 preguntas abiertas basadas en cada noticia. Mezcla preguntas técnicas con preguntas de interés general.',
        'Encuentra analogías simples para conceptos complejos. Usa comparaciones de la vida cotidiana para explicar temas técnicos. Crea metáforas visuales fáciles de entender. Adapta las analogías al público general. Mantiene el sentido original de la noticia pero la hace más accesible.'
    ]
})

agentes_df

class AgenteTitularesImpactantes:
    """Agente 1: Titulares Impactantes Contextuales"""

    def __init__(self):
        self.plantillas = {
            "pais": [
                "¡Impactante! {} - Esto cambia todo",
                "Descubre la verdad sobre {} que nadie te había contado",
                "{}: Lo que las autoridades no quieren que sepas",
                "¿Sabías esto sobre {}? Te sorprenderá",
                "{}: El secreto mejor guardado revelado"
            ],
            "mundo": [
                "Alerta global: {} - Así afecta a Chile",
                "{}: La revelación que sacude al mundo",
                "Internacional: {} - Lo que realmente significa",
                "{}: La historia detrás de las noticias",
                "Mundo: {} - Una perspectiva única"
            ],
            "deportes": [
                "¡Increíble! {} - No vas a creer lo que pasó",
                "{}: El momento que cambió todo",
                "Deportes: {} - La jugada maestra revelada",
                "{}: Lo que nunca viste en televisión",
                "¡Insólito! {} - Así se vivió el momento clave"
            ],
            "cultura": [
                "{}: El detrás de cámaras que todos esperaban",
                "Cultura: {} - La revelación exclusiva",
                "{}: Lo que no sabías sobre este evento",
                "Arte y espectáculos: {} - Todo lo que debes saber",
                "{}: El secreto mejor guardado de la industria"
            ],
            "negocios": [
                "Economía: {} - Lo que significa para tu bolsillo",
                "{}: El dato que cambiará tus inversiones",
                "Finanzas: {} - La oportunidad que no puedes perder",
                "{}: Lo que los expertos no te dicen",
                "Negocios: {} - La estrategia revelada"
            ],
            "servicios": [
                "Importante: {} - Afecta directamente a tu familia",
                "{}: El cambio que necesitas conocer",
                "Servicios: {} - Lo que debes hacer ahora",
                "{}: La guía completa que te ayudará",
                "Actualización: {} - Todo lo nuevo que debes saber"
            ]
        }
        self.palabras_tragedia = ["muere", "fallece", "accidente", "tragedia", "víctima", "fatal", "herido", "hospitalizado"]
        #Evitamos hacer clickbait en noticias negativas..

    def procesar_noticia(self, titulo, cuerpo, categoria):
        if any(palabra in titulo.lower() or palabra in cuerpo.lower() for palabra in self.palabras_tragedia):
            return f"{titulo} - Información actualizada"

        if categoria not in self.plantillas:
            categoria = "pais"

        plantilla = random.choice(self.plantillas[categoria])
        palabras = titulo.split()
        titular_reducido = " ".join(palabras[:8]) + "..." if len(palabras) > 8 else titulo

        return plantilla.format(titular_reducido)


class AgenteAnalistaDatos:
    """Agente 2: Analista de Datos Estadísticos"""

    def __init__(self):
        #Cargar modelo de spaCy para español
        try:
            self.nlp = spacy.load("es_core_news_sm")
        except:
            #Si no esta instalado, usar un enfoque más simple
            self.nlp = None
            print("spaCy no disponible, usando análisis básico")

        self.patrones_numericos = [
            (r'\$\s*(\d+[\d.,]*)', 'monetario'),
            (r'(\d+[\d.,]*)\s*%', 'porcentaje'),
            (r'\b(\d{1,2}[-/]\d{1,2}[-/]\d{2,4})\b', 'fecha'),
            (r'\b(\d+)[\s]*(millones|millón|miles|mil|billones|billón)\b', 'magnitud'),
            (r'\b(\d+[\d.,]+)\b', 'decimal'),
            (r'\b(\d+)\b', 'entero'),
        ]

    def extraer_datos_contextualizados(self, texto, titulo):
        """Extrae datos con su contexto semántico"""
        datos = []

        for patron, tipo in self.patrones_numericos:
            matches = re.finditer(patron, texto, re.IGNORECASE)
            for match in matches:
                valor = match.group(1)
                contexto = self.extraer_contexto_semantico(texto, match.start(), match.end())
                significado = self.interpretar_significado(valor, tipo, contexto, titulo)

                datos.append({
                    'valor': valor,
                    'tipo': tipo,
                    'contexto': contexto,
                    'significado': significado,
                    'relevancia': self.calcular_relevancia(valor, tipo, contexto)
                })

        return sorted(datos, key=lambda x: x['relevancia'], reverse=True)

    def extraer_contexto_semantico(self, texto, inicio, fin):
        """Extrae el contexto semántico alrededor del número"""
        margen = 100
        inicio_ctx = max(0, inicio - margen)
        fin_ctx = min(len(texto), fin + margen)
        contexto_texto = texto[inicio_ctx:fin_ctx]

        if self.nlp:
            doc = self.nlp(contexto_texto)
            #Extraer sustantivos y verbos clave alrededor del número
            palabras_clave = [
                token.text for token in doc
                if token.pos_ in ['NOUN', 'VERB', 'PROPN'] and len(token.text) > 3
            ]
            return " ".join(palabras_clave[:5])

        return contexto_texto

    def interpretar_significado(self, valor, tipo, contexto, titulo):
        """Interpreta el significado del dato en el contexto de la noticia"""
        titulo_lower = titulo.lower()
        contexto_lower = contexto.lower()

        try:
            valor_numerico = float(valor.replace('.', '').replace(',', '.'))
        except:
            valor_numerico = None

        #Análisis para películas y series (basado en tu ejemplo)
        if any(palabra in titulo_lower for palabra in ['película', 'serie', 'estreno', 'disney', 'animación']):
            if tipo == 'entero' and valor_numerico:
                if valor_numerico > 2000:  #Probablemente año
                    return f"Año de estreno: {valor}"
                elif 1 <= valor_numerico <= 10:  #Probablemente num de secuela
                    return f"Secuela número: {valor}"

        #Análisis para fechas
        if tipo == 'fecha':
            try:
                fecha = parser.parse(valor)
                return f"Fecha importante: {fecha.strftime('%d/%m/%Y')}"
            except:
                return f"Fecha referencia: {valor}"

        #Análisis para montos monetarios
        if tipo == 'monetario':
            if valor_numerico:
                if valor_numerico > 1000000:
                    return f"Gran inversión: ${valor_numerico:,.0f}"
                else:
                    return f"Monto específico: ${valor_numerico:,.0f}"

        #Análisis para porcentajes
        if tipo == 'porcentaje':
            if valor_numerico:
                if valor_numerico > 50:
                    return f"Mayoría significativa: {valor_numerico}%"
                else:
                    return f"Porcentaje relevante: {valor_numerico}%"

        return f"Dato de {tipo}: {valor}"

    def calcular_relevancia(self, valor, tipo, contexto):
        """Calcula la relevancia del dato"""
        relevancia = 0

        #Los montos monetarios son más relevantes
        if tipo == 'monetario':
            relevancia += 3
        elif tipo == 'porcentaje':
            relevancia += 2
        elif tipo == 'fecha':
            relevancia += 1

        #Valores grandes son más relevantes
        try:
            valor_num = float(valor.replace('.', '').replace(',', '.'))
            if valor_num > 1000:
                relevancia += 2
            elif valor_num > 100:
                relevancia += 1
        except:
            pass

        #Contextos específicos aumentan relevancia
        palabras_relevantes = ['millones', 'billones', 'inversión', 'presupuesto', 'crecimiento', 'aumento']
        if any(palabra in contexto.lower() for palabra in palabras_relevantes):
            relevancia += 2

        return relevancia

    def generar_analisis_narrativo(self, datos, titulo, cuerpo):
        """Genera un análisis narrativo contextualizado"""
        if not datos:
            return "No se encontraron datos numéricos significativos para analizar."

        #Filtrar datos más relevantes
        datos_relevantes = [d for d in datos if d['relevancia'] >= 2][:5]

        analisis = f"ANÁLISIS ESTADÍSTICO CONTEXTUALIZADO: {titulo}\n\n"
        analisis += "**Datos más significativos identificados:**\n\n"

        for i, dato in enumerate(datos_relevantes, 1):
            analisis += f"{i}. **{dato['valor']}** - {dato['significado']}\n"
            analisis += f"   Contexto: {dato['contexto'][:100]}...\n\n"

        #Análisis de tendencias
        analisis += "**Análisis de tendencias:**\n"
        analisis += self.analizar_tendencias(datos_relevantes, titulo)

        #Recomendaciones específicas
        analisis += "\n**Recomendaciones de interpretación:**\n"
        analisis += self.generar_recomendaciones_avanzadas(datos_relevantes, titulo, cuerpo)

        return analisis

    def analizar_tendencias(self, datos, titulo):
        """Analiza tendencias en los datos"""
        tendencias = []

        #Buscar años de estreno (para noticias de entretenimiento)
        años = []
        for dato in datos:
            try:
                valor = float(dato['valor'])
                if 2000 <= valor <= 2030:  #Rango de años plausible
                    años.append(valor)
            except:
                continue

        if años:
            tendencias.append(f"- Período de estrenos: {min(años)}-{max(años)}")

        #Buscar secuencias numéricas
        numeros = []
        for dato in datos:
            try:
                if dato['tipo'] == 'entero':
                    valor = float(dato['valor'])
                    if 1 <= valor <= 20:  #Números de secuelas
                        numeros.append(valor)
            except:
                continue

        if numeros:
            tendencias.append(f"- Secuencias: {sorted(numeros)}")

        if not tendencias:
            return "No se identificaron patrones de tendencia claros en los datos."

        return "\n".join(tendencias)

    def generar_recomendaciones_avanzadas(self, datos, titulo, cuerpo):
        """Genera recomendaciones contextualizadas"""
        recomendaciones = []
        titulo_lower = titulo.lower()

        #Para noticias de entretenimiento
        if any(palabra in titulo_lower for palabra in ['película', 'serie', 'disney', 'estreno', 'animación']):
            recomendaciones.append("• Comparar con el desempeño de secuelas anteriores")
            recomendaciones.append("• Analizar el spacing temporal entre entregas de la franquicia")
            recomendaciones.append("• Evaluar el potencial de taquilla basado en números anteriores")

        #Para noticias económicas
        if any(palabra in titulo_lower for palabra in ['inversión', 'presupuesto', 'economía', 'millones', 'billones']):
            recomendaciones.append("• Comparar con presupuestos históricos del sector")
            recomendaciones.append("• Analizar el ROI potencial de la inversión")
            recomendaciones.append("• Evaluar impacto en indicadores macroeconómicos")

        if not recomendaciones:
            recomendaciones.append("• Contextualizar los datos con benchmarks de la industria")
            recomendaciones.append("• Analizar tendencia histórica de valores similares")
            recomendaciones.append("• Evaluar impacto potencial en stakeholders relevantes")

        return "\n".join(recomendaciones)

    def procesar_noticia(self, titulo, cuerpo, categoria):
        """Procesa una noticia y genera un análisis estadístico avanzado"""
        #Extraer datos contextualizados
        datos = self.extraer_datos_contextualizados(f"{titulo} {cuerpo}", titulo)

        #Generar análisis narrativo
        return self.generar_analisis_narrativo(datos, titulo, cuerpo)

class AgenteHashtagero:
    """Agente 3: Hashtagero Estratégico"""

    def __init__(self):
        self.hashtags_por_categoria = {
            "pais": ["#Chile", "#Actualidad", "#Noticias", "#Política"],
            "mundo": ["#Internacional", "#Mundo", "#Global"],
            "deportes": ["#Deportes", "#Fútbol", "#DeporteNacional"],
            "cultura": ["#Cultura", "#Espectáculos", "#Arte"],
            "negocios": ["#Economía", "#Negocios", "#Finanzas"],
            "servicios": ["#Servicios", "#Utilidad", "#InformaciónÚtil"]
        }
        self.hashtags_emociones = {
            "positivo": ["#BuenasNoticias", "#Optimismo"],
            "neutral": ["#Información", "#Actualización"],
            "negativo": ["#Alerta", "#Importante"]
        }

    def determinar_tono(self, texto):
        texto = texto.lower()
        positivas = ["éxito", "gana", "mejora", "crece", "avance"]
        negativas = ["crisis", "problema", "alerta", "peligro", "conflicto"]

        if any(palabra in texto for palabra in positivas):
            return "positivo"
        elif any(palabra in texto for palabra in negativas):
            return "negativo"
        return "neutral"

    def procesar_noticia(self, titulo, cuerpo, categoria):
        tono = self.determinar_tono(titulo + " " + cuerpo)

        hashtags_categoria = self.hashtags_por_categoria.get(categoria, ["#Noticias"])
        hashtags_emocion = self.hashtags_emociones.get(tono, ["#Información"])

        #Extraer palabras clave para hashtags adicionales
        palabras_clave = [palabra for palabra in titulo.split() if len(palabra) > 4]
        hashtags_personalizados = ["#" + palabra for palabra in palabras_clave[:3]]

        todos_hashtags = hashtags_categoria + hashtags_emocion + hashtags_personalizados
        return " ".join(todos_hashtags[:7])  #Limitar a 7 hashtags


class AgentePreguntador:
    """Agente 4: Preguntador Curioso"""

    def __init__(self):
        #Modelo para generación de preguntas en español
        try:
            self.question_generator = pipeline(
                "text2text-generation",
                model="mrm8488/t5-base-finetuned-question-generation-ap",
                tokenizer="mrm8488/t5-base-finetuned-question-generation-ap"
            )
        except:
            self.question_generator = None
            print("Modelo de generación de preguntas no disponible")

        #Modelo para embeddings semánticos
        try:
            self.embedder = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
        except:
            self.embedder = None
            print("Modelo de embeddings no disponible")

        #Modelo para resumen y extracción de keyphrases
        try:
            self.summarizer = pipeline("summarization", model="IIC/mt5-spanish-mlsum")
        except:
            self.summarizer = None

        self.preguntas_genéricas = [
            "¿Qué impacto tendrá esto?",
            "¿Cuáles son las causas?",
            "¿Qué soluciones existen?",
            "¿Cómo afecta esto a las personas?",
            "¿Qué viene después?"
        ]

    def extraer_oraciones_clave(self, texto, num_oraciones=5):
        """Extrae las oraciones más importantes del texto"""
        if self.embedder and len(texto.split()) > 50:
            try:
                oraciones = re.split(r'[.!?]', texto)
                oraciones = [o.strip() for o in oraciones if len(o.split()) > 5]

                #Embeddings de oraciones
                embeddings = self.embedder.encode(oraciones)

                #Calc importancia (similitud con el título implícito)
                doc_embedding = self.embedder.encode([texto[:200]])
                similitudes = util.pytorch_cos_sim(embeddings, doc_embedding)

                #Seleccionar oraciones más relevantes
                indices_importantes = np.argsort(similitudes.cpu().numpy().flatten())[-num_oraciones:]
                return [oraciones[i] for i in indices_importantes]
            except:
                pass

        return texto.split('.')[:num_oraciones]

    def generar_preguntas_especificas(self, oracion, contexto):
        """Genera preguntas específicas desde una oración"""
        if self.question_generator:
            try:
                #Formato para el modelo de generación de preguntas
                input_text = f"generate question: {oracion} context: {contexto[:500]}"

                result = self.question_generator(
                    input_text,
                    max_length=64,
                    num_return_sequences=2,
                    num_beams=5,
                    early_stopping=True
                )

                preguntas = [r['generated_text'].strip() for r in result]
                return [p for p in preguntas if p.endswith('?') and len(p) > 10]
            except:
                pass

        return self.generar_preguntas_fallback(oracion)

    def generar_preguntas_fallback(self, oracion):
        """Genera preguntas usando reglas cuando no hay modelo"""
        palabras = oracion.split()
        preguntas = []

        #Identificar elementos para preguntar
        if len(palabras) > 8:
            #Pregunta sobre el sujeto
            if ':' in oracion:
                sujeto = oracion.split(':')[0]
                preguntas.append(f"¿Qué papel juega {sujeto} en esta situación?")

            #Pregunta sobre num o fechas
            numeros = re.findall(r'\b\d+\b', oracion)
            if numeros:
                preguntas.append(f"¿Qué significan estos números: {', '.join(numeros[:2])}?")

            #Pregunta sobre acciones
            verbos = ['anunció', 'confirmó', 'reveló', 'informó', 'señaló']
            for verbo in verbos:
                if verbo in oracion:
                    preguntas.append(f"¿Qué detalles hay sobre este {verbo}?")

        return preguntas[:2]

    def diversificar_preguntas(self, preguntas, similitud_umbral=0.7):
        """Elimina preguntas redundantes usando similitud semántica"""
        if not preguntas or not self.embedder:
            return preguntas

        preguntas_unicas = []
        embeddings = self.embedder.encode(preguntas)

        for i, pregunta in enumerate(preguntas):
            es_unica = True
            for pregunta_unica in preguntas_unicas:
                #Calc similitud semántica
                emb_unica = self.embedder.encode([pregunta_unica])
                similitud = util.pytorch_cos_sim(embeddings[i], emb_unica).item()

                if similitud > similitud_umbral:
                    es_unica = False
                    break

            if es_unica:
                preguntas_unicas.append(pregunta)

        return preguntas_unicas

    def adaptar_preguntas_al_contexto(self, preguntas, titulo, categoria):
        """Adapta las preguntas al contexto específico de la noticia"""
        preguntas_adaptadas = []
        titulo_lower = titulo.lower()

        for pregunta in preguntas:
            #Personalizar según categoría
            if categoria == 'cultura':
                pregunta = pregunta.replace('situación', 'evento cultural')
                pregunta = pregunta.replace('impacto', 'valor cultural')
            elif categoria == 'tecnología':
                pregunta = pregunta.replace('situación', 'desarrollo tecnológico')
                pregunta = pregunta.replace('impacto', 'innovación')

            #Personalizar según palabras clave del título
            if 'guía' in titulo_lower and 'panoramas' in titulo_lower:
                pregunta = pregunta.replace('esto', 'esta guía de panoramas')
            elif 'apple' in titulo_lower or 'google' in titulo_lower:
                pregunta = pregunta.replace('esto', 'este anuncio tecnológico')
            elif 'boric' in titulo_lower or 'presidente' in titulo_lower:
                pregunta = pregunta.replace('esto', 'este acto presidencial')

            preguntas_adaptadas.append(pregunta)

        return preguntas_adaptadas

    def procesar_noticia(self, titulo, cuerpo, categoria):
        """Procesa una noticia y genera preguntas específicas y únicas"""
        #Extraer oraciones clave
        oraciones_clave = self.extraer_oraciones_clave(cuerpo, 3)

        #Generar preguntas desde cada oración clave
        todas_preguntas = []
        for oracion in oraciones_clave:
            preguntas_oracion = self.generar_preguntas_especificas(oracion, cuerpo)
            todas_preguntas.extend(preguntas_oracion)

        #Si no se generaron preguntas, usar fallback
        if not todas_preguntas:
            todas_preguntas = self.generar_preguntas_fallback(cuerpo[:200])

        #Diversificar preguntas
        preguntas_diversas = self.diversificar_preguntas(todas_preguntas)

        #Adaptar al contexto
        preguntas_contextualizadas = self.adaptar_preguntas_al_contexto(
            preguntas_diversas, titulo, categoria
        )

        #Seleccionar las 3 mejores
        preguntas_finales = preguntas_contextualizadas[:3]

        #Formatear resultado
        resultado = f"SOBRE: {titulo}\n\n"
        resultado += "PREGUNTAS CLAVE PARA PROFUNDIZAR:\n"
        for i, pregunta in enumerate(preguntas_finales, 1):
            resultado += f"{i}. {pregunta}\n"

        return resultado


class AgenteAnalogador:
    """Agente 5: Analogador Creativo"""

    def __init__(self):
        #Modelo para generación de analogías en español
        try:
            self.analogy_generator = pipeline(
                "text2text-generation",
                model="facebook/bart-large",
                tokenizer="facebook/bart-large"
            )
        except:
            self.analogy_generator = None
            print("Modelo de generación de analogías no disponible")

        #Modelo para embeddings semánticos
        try:
            self.embedder = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
        except:
            self.embedder = None
            print("Modelo de embeddings no disponible")

        #"Base de datos" de analogías por categoría
        self.analogias_por_tema = {
            "justicia": [
                "es como un juego de ajedrez donde cada movimiento debe ser estratégico",
                "se parece a una balanza que busca equilibrar derechos y responsabilidades",
                "es como una investigación detectivesca que busca encontrar la verdad"
            ],
            "medio_ambiente": [
                "es como cuidar un jardín que requiere atención constante",
                "se parece a respirar aire fresco después de mucho tiempo",
                "es como ver florecer la naturaleza después de un invierno largo"
            ],
            "economía": [
                "es como armar un rompecabezas financiero complejo",
                "se parece a navegar en aguas económicas turbulentas",
                "es como construir una casa con cimientos sólidos"
            ],
            "política": [
                "es como un tablero de ajedrez geopolítico",
                "se parece a una danza diplomática cuidadosamente coreografiada",
                "es como tejer una red de relaciones internacionales"
            ],
            "tecnología": [
                "es como abrir una caja de pandora de innovaciones",
                "se parece a surfear una ola de transformación digital",
                "es como construir puentes hacia el futuro"
            ],
            "salud": [
                "es como curar heridas en el sistema de salud",
                "se parece a encontrar la medicina adecuada para una enfermedad",
                "es como fortalecer el sistema inmunológico social"
            ]
        }

        self.palabras_clave_temas = {
            "justicia": ["fiscal", "juez", "tribunal", "ley", "investigación", "delito"],
            "medio_ambiente": ["aire", "calidad", "contaminación", "medio ambiente", "ecología", "sostenible"],
            "economía": ["empleo", "desempleo", "económico", "mercado", "finanzas", "crecimiento"],
            "política": ["oposición", "gobierno", "presidente", "político", "ley", "debate"],
            "tecnología": ["digital", "tecnología", "innovación", "app", "internet", "plataforma"],
            "salud": ["hospital", "médico", "salud", "paciente", "enfermedad", "tratamiento"]
        }

    def identificar_tema_principal(self, texto):
        """Identifica el tema principal de la noticia"""
        texto_lower = texto.lower()
        scores = {}

        for tema, palabras in self.palabras_clave_temas.items():
            score = sum(1 for palabra in palabras if palabra in texto_lower)
            scores[tema] = score

        if scores:
            return max(scores.items(), key=lambda x: x[1])[0]
        return "general"

    def generar_analogia_especifica(self, titulo, cuerpo, tema):
        """Genera una analogía específica usando el modelo"""
        try:
            #Preparar el prompt para el modelo
            prompt = f"Genera una analogía creativa en español para esta noticia: {titulo}. "
            prompt += f"Contexto: {cuerpo[:200]}... "
            prompt += "La analogía debe ser comprensible y relacionada con la vida cotidiana."

            #Generar analogía
            result = self.analogy_generator(
                prompt,
                max_length=100,
                num_return_sequences=1,
                num_beams=3,
                early_stopping=True
            )

            analogia = result[0]['generated_text'].strip()

            #Limpiar y formatear la analogía
            analogia = self.limpiar_analogia(analogia)
            return analogia

        except Exception as e:
            print(f"Error generando analogía: {e}")
            return self.analogia_fallback(tema)

    def limpiar_analogia(self, analogia):
        """Limpia y formatea la analogía generada"""
        #Remover el prompt si esta presente
        analogia = re.sub(r'Genera una analogía.*?:', '', analogia)
        analogia = re.sub(r'Contexto:.*?\.', '', analogia)

        #Asegurar que sea una frase completa
        if not analogia.endswith('.'):
            analogia += '.'

        #Capitalizar
        analogia = analogia.capitalize()

        return analogia.strip()

    def analogia_fallback(self, tema):
        """Analogía de fallback basada en el tema"""
        if tema in self.analogias_por_tema:
            return random.choice(self.analogias_por_tema[tema])

        #Analogías generales
        analogias_generales = [
            "es como armar un rompecabezas donde cada pieza tiene su lugar",
            "se parece a navegar en aguas desconocidas buscando el mejor camino",
            "es como encontrar la pieza faltante de un misterio complejo",
            "se parece a ver el amanecer después de una noche larga",
            "es como tejer una red de soluciones interconectadas"
        ]
        return random.choice(analogias_generales)

    def crear_analogia_contextual(self, titulo, cuerpo):
        """Crea una analogía contextualizada manualmente"""
        texto_completo = f"{titulo} {cuerpo}".lower()

        #Det características específicas
        if any(palabra in texto_completo for palabra in ['justicia', 'fiscal', 'juez']):
            return "es como un delicado equilibrio entre la ley y la equidad, donde cada decisión puede cambiar el curso de los eventos"

        elif any(palabra in texto_completo for palabra in ['aire', 'calidad', 'contaminación']):
            return "se parece a respirar profundamente después de años de esfuerzos por limpiar el ambiente, mostrando que la perseverancia da frutos"

        elif any(palabra in texto_completo for palabra in ['empleo', 'desempleo', 'económico']):
            return "es como construir un puente sobre aguas turbulentas, donde cada puesto de trabajo es un pilote esencial para la estabilidad"

        elif any(palabra in texto_completo for palabra in ['tecnología', 'digital', 'innovación']):
            return "es como abrir una ventana al futuro, donde cada avance tecnológico nos acerca a posibilidades antes inimaginables"

        elif any(palabra in texto_completo for palabra in ['salud', 'hospital', 'médico']):
            return "se parece a fortalecer los cimientos de un sistema que cura y protege, donde cada mejora salva vidas y construye bienestar"

        #Analogía genérica pero contextualizada
        return "es como encontrar el hilo conductor en una madeja compleja, donde cada elemento se conecta para formar una comprensión más completa"

    def procesar_noticia(self, titulo, cuerpo, categoria):
        """Procesa una noticia y genera una analogía creativa"""
        #Identificar tema principal
        tema = self.identificar_tema_principal(f"{titulo} {cuerpo}")

        #Generar analogía
        if self.analogy_generator:
            try:
                analogia = self.generar_analogia_especifica(titulo, cuerpo, tema)
            except:
                analogia = self.crear_analogia_contextual(titulo, cuerpo)
        else:
            analogia = self.crear_analogia_contextual(titulo, cuerpo)

        #Formatear resultado
        return f"'{titulo}' {analogia} Esta analogía nos ayuda a visualizar mejor la situación."


In [8]:
# Ejecución de cada agente sobre el corpus

#Cargar el corpus de noticias
df = pd.read_csv('datos/cnnchile_noticias.csv')
total_noticias = len(df)
noticias_por_agente = 10

def procesar_con_agente(agente, nombre_agente, indices_noticias, df):
    """Procesa noticias específicas con un agente y guarda los resultados en un CSV"""
    resultados = []

    for idx in indices_noticias:
        if idx < len(df):
            row = df.iloc[idx]
            resultado = agente.procesar_noticia(row['titulo'], row['cuerpo'], row['categoria'])
            resultados.append({
                'indice_original': idx,
                'titulo_original': row['titulo'],
                'categoria': row['categoria'],
                'url': row['url'],
                'resultado': resultado
            })

    #Crear DataFrame y guardar
    df_resultados = pd.DataFrame(resultados)
    archivo_salida = f"datos/agente_{nombre_agente.lower().replace(' ', '_')}.csv"
    df_resultados.to_csv(archivo_salida, index=False, encoding='utf-8-sig')

    print(f"{nombre_agente}: {len(resultados)} noticias procesadas -> {archivo_salida}")
    return df_resultados

#Asignar indices de noticias diferentes para cada agente
if total_noticias >= 50:
    #Si hay suficientes noticias, asignar rangos distintos
    indices_titulares = list(range(0, noticias_por_agente))
    indices_analista = list(range(noticias_por_agente, 2 * noticias_por_agente))
    indices_hashtagero = list(range(2 * noticias_por_agente, 3 * noticias_por_agente))
    indices_preguntador = list(range(3 * noticias_por_agente, 4 * noticias_por_agente))
    indices_analogador = list(range(4 * noticias_por_agente, 5 * noticias_por_agente))
else:
    #Si no hay suficientes noticias, seleccionar aleatoriamente con semillas diferentes
    random.seed(42)  #Semilla fija para reproducibilidad
    todos_indices = list(range(total_noticias))

    indices_titulares = random.sample(todos_indices, min(noticias_por_agente, total_noticias))
    #Eliminar los indices ya seleccionados para evitar superposición
    indices_restantes = [i for i in todos_indices if i not in indices_titulares]

    indices_analista = random.sample(indices_restantes, min(noticias_por_agente, len(indices_restantes)))
    indices_restantes = [i for i in indices_restantes if i not in indices_analista]

    indices_hashtagero = random.sample(indices_restantes, min(noticias_por_agente, len(indices_restantes)))
    indices_restantes = [i for i in indices_restantes if i not in indices_hashtagero]

    indices_preguntador = random.sample(indices_restantes, min(noticias_por_agente, len(indices_restantes)))
    indices_restantes = [i for i in indices_restantes if i not in indices_preguntador]

    indices_analogador = random.sample(indices_restantes, min(noticias_por_agente, len(indices_restantes)))

#Procesar con cada agente
print("Procesando noticias con los 5 agentes...")
print("=" * 50)

#Inicializar agentes
agente_titulares = AgenteTitularesImpactantes()
agente_analista = AgenteAnalistaDatos()
agente_hashtagero = AgenteHashtagero()
agente_preguntador = AgentePreguntador()
agente_analogador = AgenteAnalogador()

#Agente 1: Titulares Impactantes
df_titulares = procesar_con_agente(agente_titulares, "Titulares Impactantes", indices_titulares, df)

#Agente 2: Analista de Datos
df_analista = procesar_con_agente(agente_analista, "Analista de Datos", indices_analista, df)

#Agente 3: Hashtagero Estrategico
df_hashtagero = procesar_con_agente(agente_hashtagero, "Hashtagero Estrategico", indices_hashtagero, df)

#Agente 4: Preguntador Curioso
df_preguntador = procesar_con_agente(agente_preguntador, "Preguntador Curioso", indices_preguntador, df)

#Agente 5: Analogador Creativo
df_analogador = procesar_con_agente(agente_analogador, "Analogador Creativo", indices_analogador, df)

print("=" * 50)
print("Procesamiento completado. Archivos guardados en la carpeta 'datos'")

#Verificar que no hay superposicion de indices
todos_indices_procesados = set(indices_titulares + indices_analista + indices_hashtagero + indices_preguntador + indices_analogador)
print(f"Total de noticias únicas procesadas: {len(todos_indices_procesados)}")
print(f"Total de noticias en el corpus: {total_noticias}")

#Mostrar ejemplos de cada agente
print("\nEjemplos de resultados:")
print("=" * 50)

def mostrar_ejemplo(df_agente, nombre_agente):
    if len(df_agente) > 0:
        print(f"\n{nombre_agente}:")
        print(f"Índice de noticia: {df_agente['indice_original'].iloc[0]}")
        print(f"Título original: {df_agente['titulo_original'].iloc[0][:50]}...")
        print(f"Resultado: {df_agente['resultado'].iloc[0][:100]}...")

mostrar_ejemplo(df_titulares, "1. Titulares Impactantes")
mostrar_ejemplo(df_analista, "2. Analista de Datos")
mostrar_ejemplo(df_hashtagero, "3. Hashtagero Estratégico")
mostrar_ejemplo(df_preguntador, "4. Preguntador Curioso")
mostrar_ejemplo(df_analogador, "5. Analogador Creativo")

Procesando noticias con los 5 agentes...
spaCy no disponible, usando análisis básico


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

Titulares Impactantes: 10 noticias procesadas -> datos/agente_titulares_impactantes.csv
Analista de Datos: 10 noticias procesadas -> datos/agente_analista_de_datos.csv
Hashtagero Estrategico: 10 noticias procesadas -> datos/agente_hashtagero_estrategico.csv


Both `max_new_tokens` (=256) and `max_length`(=64) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=64) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=64) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=64) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

Preguntador Curioso: 10 noticias procesadas -> datos/agente_preguntador_curioso.csv


Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Analogador Creativo: 10 noticias procesadas -> datos/agente_analogador_creativo.csv
Procesamiento completado. Archivos guardados en la carpeta 'datos'
Total de noticias únicas procesadas: 50
Total de noticias en el corpus: 130

Ejemplos de resultados:

1. Titulares Impactantes:
Índice de noticia: 0
Título original: Cómo Pedro Pascal se convirtió en el actor más cot...
Resultado: Cultura: Cómo Pedro Pascal se convirtió en el actor... - La revelación exclusiva...

2. Analista de Datos:
Índice de noticia: 10
Título original: Superclásico 198: Universidad de Chile sufre la ex...
Resultado: ANÁLISIS ESTADÍSTICO CONTEXTUALIZADO: Superclásico 198: Universidad de Chile sufre la expulsión de C...

3. Hashtagero Estratégico:
Índice de noticia: 20
Título original: “Me desmayé por estrés”: Endler revela secuelas de...
Resultado: #Deportes #Fútbol #DeporteNacional #BuenasNoticias #Optimismo #desmayé #estrés”:...

4. Preguntador Curioso:
Índice de noticia: 30
Título original: “Volveremos siendo aún 

## Parte 3: Generación de portal de noticias en HTML


### Instrucciones:

- Crea un sitio web HTML básico (una sola página o varias secciones).
- Usa los textos generados por los agentes para crear un portal de noticias HTML.
- Muestra las noticias generadas por tus agentes, agrupadas por autor (agente).
- Incluye:
    - Título.
    - Fecha.
    - Nombre del agente.
    - Cuerpo del texto generado.
    - Imágenes (opcional).
- Puedes estilizar con CSS básico si lo deseas, pero se evalúa el contenido funcional.
- Puedes generar el HTML directamente desde Python con `Jinja2`, o usando templates simples.

In [9]:
# Código para generar HTML con noticias agrupadas por agente

#Crear carpeta 'sitio' si no existe
if not os.path.exists('sitio'):
    os.makedirs('sitio')

#Cargar los datos de todos los agentes
def cargar_datos_agentes():
    agentes = {
        'titulares_impactantes': pd.read_csv('datos/agente_titulares_impactantes.csv'),
        'analista_datos': pd.read_csv('datos/agente_analista_de_datos.csv'),
        'hashtagero_estrategico': pd.read_csv('datos/agente_hashtagero_estrategico.csv'),
        'preguntador_curioso': pd.read_csv('datos/agente_preguntador_curioso.csv'),
        'analogador_creativo': pd.read_csv('datos/agente_analogador_creativo.csv')
    }

    #Cargar tmbn el corpus original para obtener fechas completas
    corpus_original = pd.read_csv('datos/cnnchile_noticias.csv')

    #Agregar fechas a los datos de los agentes
    for nombre_agente, df_agente in agentes.items():
        if 'indice_original' in df_agente.columns:
            df_agente['fecha'] = df_agente['indice_original'].apply(
                lambda idx: corpus_original.iloc[idx]['fecha'] if idx < len(corpus_original) else 'Fecha no disponible'
            )
        else:
            df_agente['fecha'] = 'Fecha no disponible'

    return agentes

#Plantilla HTML base usando Jinja2
template_html = """
<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Portal de Noticias Generadas por Agentes</title>
    <style>
        * {
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }

        body {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            line-height: 1.6;
            color: #333;
            background-color: #f4f4f4;
        }

        .container {
            max-width: 1200px;
            margin: 0 auto;
            padding: 20px;
        }

        header {
            background-color: #2c3e50;
            color: white;
            padding: 20px 0;
            text-align: center;
            margin-bottom: 30px;
        }

        h1 {
            font-size: 2.5rem;
            margin-bottom: 10px;
        }

        .subtitle {
            font-size: 1.2rem;
            opacity: 0.8;
        }

        .agente-section {
            background-color: white;
            border-radius: 8px;
            box-shadow: 0 2px 10px rgba(0, 0, 0, 0.1);
            margin-bottom: 30px;
            overflow: hidden;
        }

        .agente-header {
            background-color: #3498db;
            color: white;
            padding: 15px 20px;
            display: flex;
            justify-content: space-between;
            align-items: center;
        }

        .agente-title {
            font-size: 1.5rem;
            font-weight: bold;
        }

        .agente-desc {
            font-size: 1rem;
            opacity: 0.9;
        }

        .noticias-container {
            padding: 20px;
        }

        .noticia {
            border-bottom: 1px solid #eee;
            padding: 15px 0;
        }

        .noticia:last-child {
            border-bottom: none;
        }

        .noticia-titulo {
            font-size: 1.2rem;
            color: #2c3e50;
            margin-bottom: 10px;
        }

        .noticia-meta {
            font-size: 0.9rem;
            color: #7f8c8d;
            margin-bottom: 10px;
        }

        .noticia-contenido {
            margin-bottom: 15px;
            line-height: 1.6;
        }

        .noticia-url {
            display: inline-block;
            color: #3498db;
            text-decoration: none;
            font-size: 0.9rem;
        }

        .noticia-url:hover {
            text-decoration: underline;
        }

        footer {
            text-align: center;
            margin-top: 40px;
            padding: 20px;
            color: #7f8c8d;
            font-size: 0.9rem;
        }

        @media (max-width: 768px) {
            .container {
                padding: 10px;
            }

            .agente-header {
                flex-direction: column;
                text-align: left;
            }

            .agente-title {
                margin-bottom: 5px;
            }
        }
    </style>
</head>
<body>
    <header>
        <div class="container">
            <h1>Portal de Noticias Generadas por Agentes</h1>
            <p class="subtitle">Contenido procesado automáticamente por agentes especializados</p>
        </div>
    </header>

    <div class="container">
        {% for agente in agentes %}
        <section class="agente-section">
            <div class="agente-header">
                <div>
                    <h2 class="agente-title">{{ agente.nombre }}</h2>
                    <p class="agente-desc">{{ agente.descripcion }}</p>
                </div>
                <span class="contador-noticias">{{ agente.noticias|length }} noticias</span>
            </div>

            <div class="noticias-container">
                {% for noticia in agente.noticias %}
                <article class="noticia">
                    <h3 class="noticia-titulo">{{ noticia.titulo_original }}</h3>
                    <div class="noticia-meta">
                        <span class="fecha">{{ noticia.fecha }}</span> |
                        <span class="categoria">{{ noticia.categoria }}</span> |
                        <span class="agente">Por: {{ agente.nombre }}</span>
                    </div>
                    <div class="noticia-contenido">
                        {{ noticia.resultado }}
                    </div>
                    <a href="{{ noticia.url }}" class="noticia-url" target="_blank">Ver noticia original</a>
                </article>
                {% endfor %}
            </div>
        </section>
        {% endfor %}
    </div>

    <footer>
        <div class="container">
            <p>Portal generado automáticamente el {{ fecha_generacion }}</p>
            <p>Contenido procesado por {{ total_noticias }} noticias de {{ total_agentes }} agentes diferentes</p>
        </div>
    </footer>
</body>
</html>
"""

#Info descriptiva de cada agente
info_agentes = {
    'titulares_impactantes': {
        'nombre': 'Titulares Impactantes Contextuales',
        'descripcion': 'Genera titulares clickbait ajustados a la categoría, evitando sensacionalizar tragedias.'
    },
    'analista_datos': {
        'nombre': 'Analista de Datos Estadísticos',
        'descripcion': 'Extrae y analiza cifras, porcentajes y datos numéricos de las noticias.'
    },
    'hashtagero_estrategico': {
        'nombre': 'Hashtagero Estratégico',
        'descripcion': 'Identifica palabras clave y genera combinaciones de hashtags por categoría temática.'
    },
    'preguntador_curioso': {
        'nombre': 'Preguntador Curioso',
        'descripcion': 'Extrae preguntas abiertas basadas en aspectos de los artículos originales.'
    },
    'analogador_creativo': {
        'nombre': 'Analogador Creativo',
        'descripcion': 'Encuentra analogías simples para conceptos complejos, haciendo las noticias más accesibles.'
    }
}

#Cargar datos de los agentes
agentes_data = cargar_datos_agentes()

#Preparar datos para la plantilla
agentes_para_template = []
total_noticias = 0

for agente_id, df_agente in agentes_data.items():
    noticias = []

    for _, row in df_agente.iterrows():
        noticias.append({
            'titulo_original': row['titulo_original'],
            'categoria': row['categoria'],
            'url': row['url'],
            'fecha': row['fecha'],
            'resultado': row['resultado']
        })

    total_noticias += len(noticias)

    agentes_para_template.append({
        'id': agente_id,
        'nombre': info_agentes[agente_id]['nombre'],
        'descripcion': info_agentes[agente_id]['descripcion'],
        'noticias': noticias
    })

#Renderizar la plantilla
template = Template(template_html)
html_output = template.render(
    agentes=agentes_para_template,
    fecha_generacion=datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    total_noticias=total_noticias,
    total_agentes=len(agentes_para_template)
)

#Guardar el archivo HTML
with open('sitio/portal_noticias.html', 'w', encoding='utf-8') as f:
    f.write(html_output)

print("Portal de noticias generado exitosamente en: sitio/portal_noticias.html")

#Crear una pag de índice simple
indice_html = """
<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Portal de Noticias - Índice</title>
    <style>
        body {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            line-height: 1.6;
            color: #333;
            background-color: #f4f4f4;
            margin: 0;
            padding: 20px;
            display: flex;
            flex-direction: column;
            align-items: center;
            justify-content: center;
            min-height: 100vh;
            text-align: center;
        }

        h1 {
            color: #2c3e50;
            margin-bottom: 30px;
        }

        .enlace-portal {
            display: inline-block;
            background-color: #3498db;
            color: white;
            padding: 15px 30px;
            text-decoration: none;
            border-radius: 5px;
            font-size: 1.2rem;
            margin: 10px;
            transition: background-color 0.3s;
        }

        .enlace-portal:hover {
            background-color: #2980b9;
        }

        .agentes-lista {
            margin-top: 30px;
            text-align: left;
            max-width: 600px;
        }

        .agente-item {
            background-color: white;
            padding: 15px;
            margin-bottom: 10px;
            border-radius: 5px;
            box-shadow: 0 2px 5px rgba(0,0,0,0.1);
        }

        .agente-nombre {
            font-weight: bold;
            color: #2c3e50;
        }

        .agente-desc {
            color: #7f8c8d;
            font-size: 0.9rem;
        }
    </style>
</head>
<body>
    <h1>Portal de Noticias Generadas por Agentes</h1>

    <a href="portal_noticias.html" class="enlace-portal">Ver Portal Completo de Noticias</a>

    <div class="agentes-lista">
        <h2>Agentes de Contenido</h2>
        <div class="agente-item">
            <div class="agente-nombre">Titulares Impactantes Contextuales</div>
            <div class="agente-desc">Genera titulares clickbait ajustados a la categoría, evitando sensacionalizar tragedias.</div>
        </div>
        <div class="agente-item">
            <div class="agente-nombre">Analista de Datos Estadísticos</div>
            <div class="agente-desc">Extrae y analiza cifras, porcentajes y datos numéricos de las noticias.</div>
        </div>
        <div class="agente-item">
            <div class="agente-nombre">Hashtagero Estratégico</div>
            <div class="agente-desc">Identifica palabras clave y genera combinaciones de hashtags por categoría temática.</div>
        </div>
        <div class="agente-item">
            <div class="agente-nombre">Preguntador Curioso</div>
            <div class="agente-desc">Extrae preguntas abiertas basadas en aspectos de los artículos originales.</div>
        </div>
        <div class="agente-item">
            <div class="agente-nombre">Analogador Creativo</div>
            <div class="agente-desc">Encuentra analogías simples para conceptos complejos, haciendo las noticias más accesibles.</div>
        </div>
    </div>
</body>
</html>
"""

with open('sitio/index.html', 'w', encoding='utf-8') as f:
    f.write(indice_html)

print("Página de índice generada en: sitio/index.html")

Portal de noticias generado exitosamente en: sitio/portal_noticias.html
Página de índice generada en: sitio/index.html


## Reflexión final


Crea un archivo `README.md` y responde:
- ¿Cuál agente generó contenido más coherente?
- ¿Qué dificultades tuviste al automatizar la generación de noticias?
- ¿Qué herramientas o métodos usarías si lo hicieras de nuevo?


## Entregables

Un archivo comprimido `.zip` con nombre `T1_TEL351_Nombre_Apellido.zip` que contenga:

1. **Jupyter Notebook** con todo el proceso:

    - Scraping

    - Procesamiento de datos

    - Implementación de los agentes

    - Generación del sitio

2. **Carpeta** `datos/` con el corpus original y el generado por los agentes.

3. **Carpeta** `sitio/` con archivos HTML y medios asociados.

4. **Archivo** `README.md`